In [6]:
import cv2
import face_recognition
import numpy as np
import os
import pickle


In [9]:
known_face_encodings = []
known_face_names = []

dataset_path = "Dataset"

for person_name in os.listdir(dataset_path):

    person_folder = os.path.join(
        dataset_path,
        person_name
    )

    for image_name in os.listdir(person_folder):

        image_path = os.path.join(
            person_folder,
            image_name
        )

        image = face_recognition.load_image_file(
            image_path
        )

        encodings = face_recognition.face_encodings(
            image
        )

        if len(encodings) > 0:

            known_face_encodings.append(
                encodings[0]
            )

            known_face_names.append(
                person_name
            )

print("Faces Loaded:",
      len(known_face_names))

Faces Loaded: 4


In [10]:
data = {
    "encodings" : known_face_encodings,
    "names" : known_face_names 
}
with open("models/encodings.pkl","wb") as f:
    pickle.dump(data,f)
print("Saved")

Saved


In [12]:
with open(
    "models/encodings.pkl",
    "rb"
) as f:

    data = pickle.load(f)

known_face_encodings = data["encodings"]
known_face_names = data["names"]

print("Loaded")

Loaded


In [13]:
import cv2
import face_recognition
import numpy as np

# Start webcam
video_capture = cv2.VideoCapture(0)

while True:

    # Capture frame-by-frame
    ret, frame = video_capture.read()

    if not ret:
        print("Failed to access webcam")
        break

    # Convert BGR (OpenCV) to RGB (face_recognition)
    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    # Detect faces
    face_locations = face_recognition.face_locations(
        rgb_frame
    )

    # Generate encodings for detected faces
    face_encodings = face_recognition.face_encodings(
        rgb_frame,
        face_locations
    )

    # Process each face
    for (top, right, bottom, left), face_encoding in zip(
        face_locations,
        face_encodings
    ):

        name = "Unknown"

        # Compare against known faces
        matches = face_recognition.compare_faces(
            known_face_encodings,
            face_encoding
        )

        face_distances = face_recognition.face_distance(
            known_face_encodings,
            face_encoding
        )

        if len(face_distances) > 0:

            best_match_index = np.argmin(
                face_distances
            )

            if matches[best_match_index]:
                name = known_face_names[
                    best_match_index
                ]

        # Draw rectangle around face
        cv2.rectangle(
            frame,
            (left, top),
            (right, bottom),
            (0, 255, 0),
            2
        )

        # Draw name label
        cv2.rectangle(
            frame,
            (left, bottom - 35),
            (right, bottom),
            (0, 255, 0),
            cv2.FILLED
        )

        cv2.putText(
            frame,
            name,
            (left + 6, bottom - 6),
            cv2.FONT_HERSHEY_DUPLEX,
            0.8,
            (255, 255, 255),
            1
        )

    # Display resulting frame
    cv2.imshow(
        "Face Recognition",
        frame
    )

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
video_capture.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 